In [1]:
import os, json
import torch
import transformers
import pandas as pd
import numpy as np

import transformers 
from transformers import AutoTokenizer

import MeMoHF
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import (
    seed_everything,
    load_model_and_tokenizer,
    save_data_to_disk,
    load_from_disk
)

import datasets 
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

os.environ['CUDA_VISIBLE_DEVICES'] = '0'



In [2]:
from learning_evaluation import create_large_sample

In [3]:
data_path = '/home/davide/.cache/huggingface/datasets/wikipedia/20200501.en/1.0.0/009f923d9b6dd00c00c8cdc7f408f2b47f45dd4f5fb7982a21f9448f4afbe475/wikipedia-train.arrow'
data = dict(
    train=Dataset.from_file(data_path)
)

In [4]:
data['train'].select_columns('text')

Dataset({
    features: ['text'],
    num_rows: 6078422
})

In [5]:
import learning_evaluation
from learning_evaluation import create_all_datasets

# create_all_datasets(main_data_dir='training_data')


In [6]:
def convert_text_into_cfg(text):
    cfg_list = [
        (param.split('=['))
        for param in text.split(']-')
    ]
    return {
        param[0]:param[1]
        for param in cfg_list
    }

convert_text_into_cfg(os.path.basename('bla/di/bla/max_length=[1024]-d=[1024]-l=[4]-h=[4]-batch_size=[64]-save_every_k_batches=[3]-data_name=[n=1000]-seed=[42]-batch_id=[3]'))

{'max_length': '1024',
 'd': '1024',
 'l': '4',
 'h': '4',
 'batch_size': '64',
 'save_every_k_batches': '3',
 'data_name': 'n=1000',
 'seed': '42',
 'batch_id': '3]'}

In [7]:
import pandas as pd

# Sample DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
    {'lr': 0.01, 'batch_size': 64, 'optimizer': 'adam'},
])

# Dictionary with a subset of keys
partial_experiment = {'lr': 0.01, 'optimizer': 'adam'}

# Subset DataFrame to the relevant columns and compare
match = (df[partial_experiment.keys()] == pd.Series(partial_experiment)).all(axis=1)

# Check if any row matches the subset
exists = match.any()

print("Subset match exists:", exists)

Subset match exists: True


In [8]:
import pandas as pd

# Existing DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
])

# New row as a dictionary
new_row = {'lr': 0.005, 'batch_size': 128, 'optimizer': 'adam'}

# Add the new row
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

print(df)

      lr  batch_size optimizer
0  0.010          32      adam
1  0.001          64       sgd
2  0.005         128      adam


Experiments with sequence view augmentation for computing multiple sequences in parallel given [batch_size] texts

In [9]:
import torch 
import torch.nn.functional as F

# Example input
batch_size = 3
seq_len = 4
hidden_dim = 2
window_size = 2

# x = torch.randn(batch_size, seq_len, hidden_dim)  # (B, S, H)
x = torch.tensor(
    [
        [[1,2],[3,4],[5,6],[7,8]],
        [[9,10],[11,12],[13,14],[15,16]],
        [[17,18],[19,20],[21,22],[23,24]],
    ],
    dtype=torch.int32
)
y = torch.tensor(
    [
        [1,2,3,4],
        [5,6,7,8],
        [9,10,11,12],
    ],
    dtype=torch.int32
)
print(x)
print(y)

tensor([[[ 1,  2],
         [ 3,  4],
         [ 5,  6],
         [ 7,  8]],

        [[ 9, 10],
         [11, 12],
         [13, 14],
         [15, 16]],

        [[17, 18],
         [19, 20],
         [21, 22],
         [23, 24]]], dtype=torch.int32)
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]], dtype=torch.int32)


In [10]:
num_windows = seq_len - window_size + 1 
x_windows = x.permute(0, 2, 1) # (B, H, S)
# Now unfold sequence dimension (dim=2)
x_windows = x_windows.unfold(dimension=2, size=window_size, step=1) # (B, H, num_windows, window_size)
# Now bring it back to (B, num_windows, window_size, H)
x_windows = x_windows.permute(0, 2, 3, 1) # (B, num_windows, window_size, H)
# Reshape to (B * num_windows, window_size, H)
x_windows = x_windows.contiguous().view(-1, window_size, hidden_dim)
print(x_windows)

y_windows = y[:, 1:].unfold(dimension=1, size=1, step=1)
y_windows = y_windows.contiguous().view(-1, 1)
print(y_windows)

tensor([[[ 1,  2],
         [ 3,  4]],

        [[ 3,  4],
         [ 5,  6]],

        [[ 5,  6],
         [ 7,  8]],

        [[ 9, 10],
         [11, 12]],

        [[11, 12],
         [13, 14]],

        [[13, 14],
         [15, 16]],

        [[17, 18],
         [19, 20]],

        [[19, 20],
         [21, 22]],

        [[21, 22],
         [23, 24]]], dtype=torch.int32)
tensor([[ 2],
        [ 3],
        [ 4],
        [ 6],
        [ 7],
        [ 8],
        [10],
        [11],
        [12]], dtype=torch.int32)


In [11]:
new_hidden_dim = hidden_dim
final_x = x_windows.sum(dim=1)
final_x = final_x.view(batch_size, num_windows, new_hidden_dim)
final_y = y_windows.view(batch_size, -1)

print(final_x)
print(final_y)

tensor([[[ 4,  6],
         [ 8, 10],
         [12, 14]],

        [[20, 22],
         [24, 26],
         [28, 30]],

        [[36, 38],
         [40, 42],
         [44, 46]]])
tensor([[ 2,  3,  4],
        [ 6,  7,  8],
        [10, 11, 12]], dtype=torch.int32)


MEMO Debugging

In [12]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

d, h, l = 1024, 4, 2
chunk_length = 16

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
               hidden_size=d, 
               num_hidden_layers=l,
               num_attention_heads=h,
               chunk_length=chunk_length,
               bos_token_id=tokenizer.bos_token_id,
               eos_token_id=tokenizer.eos_token_id,
               pad_token_id=tokenizer.pad_token_id,
              )

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
MeMo embedding initilialization
MeMo embedding initilialization
MeMo embedding initilialization
GPU: NVIDIA RTX A6000 is available.


In [13]:
first_text = ["a b c d e f g h i j k l m n o p q"]
memo_input = tokenizer.get_text_batch_encoding(first_text * 8)

print(tokenizer.decode(memo_input['input_ids'][0]))
print(tokenizer.decode(memo_input['labels'][0]))

a b c d e f g h i j k l m n o p
 b c d e f g h i j k l m n o p q


In [14]:
# evaluation method
from MeMoHF.evaluating_memo import EvaluationUpdateNew
evaluation = EvaluationUpdateNew()

def perform_evaluation(model, tokenizer, text, starting_point=1):
    model.eval()
    batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=text)
    with torch.no_grad():
        outs, pretokenized_score = model.forward_with_loss_parallelized(
            batch_inputs=batch_inputs,
            compute_accuracy=True
        )

    # memo_input = tokenizer.get_text_batch_encoding(text=text)
    # with torch.no_grad():
    #     pretokenized_score = evaluation.check_pretokenized(
    #         model=model,
    #         tokenizer=tokenizer,
    #         input_ids=memo_input['input_ids'],
    #         starting_point=starting_point
    #     )
    loss = outs['loss']
    del outs
    model.train()
    torch.cuda.empty_cache()
    return dict(
        out_loss=loss,
        token_accuracy=pretokenized_score
    )

In [15]:
results = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=first_text*8
)
print(results)

# results_2 = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text,
#     starting_point=None
# )
# print(results_2)

{'out_loss': tensor(10.8253, device='cuda:0'), 'token_accuracy': {'accuracy': 0.0, 'correct_tokens': 0, 'tot_tokens': 136}}


In [16]:
# memorize input
model.memorize_text(memo_input)

In [17]:
# post-edit evaluation
results = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=first_text*8
)
print(results)

# results_2 = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text,
#     starting_point=None
# )
# print(results_2)

{'out_loss': tensor(10.9549, device='cuda:0'), 'token_accuracy': {'accuracy': 0.7647058963775635, 'correct_tokens': 104, 'tot_tokens': 136}}


In [18]:
# a = a + 1

In [19]:
model.save_pretrained('memo_example')
tokenizer.save_pretrained('memo_example')

model2 = MeMoForCausalLM.from_pretrained('memo_example', device_map='auto')
tokenizer2 = MeMoTokenizer.from_pretrained('memo_example')
model2

MeMo embedding initilialization
MeMo embedding initilialization
Setting pad token and pad token id = <|endoftext|>, 0


MeMoForCausalLM(
  (memo): MeMo(
    (encoder): MeMoEmbedding(50277, 1024, padding_idx=0)
    (layers): MeMoLayers(
      (0): MeMoLayer(
        (W_v_single_head): ProjectionTokens(in_features=1024, out_features=256)
        (Prj): ProjectionSequence((trasposed wrt saved one) in_features=4096, out_features=1024)
      )
      (1): MeMoLayer(
        (W_v_single_head): ProjectionTokens(in_features=1024, out_features=256)
        (Prj): ProjectionSequence((trasposed wrt saved one) in_features=4096, out_features=1024)
        (CMM): CorrelationMatrixMemory(in_features=1024, out_features=1024)
      )
    )
  )
  (lm_head): MeMoEmbedding(50277, 1024, padding_idx=0)
)

In [20]:
results = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=first_text*8
)
print(results)

{'out_loss': tensor(10.9549, device='cuda:0'), 'token_accuracy': {'accuracy': 0.7647058963775635, 'correct_tokens': 104, 'tot_tokens': 136}}


In [21]:
results = perform_evaluation(
    model=model2,
    tokenizer=tokenizer2,
    text=first_text*8
)
print(results)

{'out_loss': tensor(10.9549, device='cuda:0'), 'token_accuracy': {'accuracy': 0.7647058963775635, 'correct_tokens': 104, 'tot_tokens': 136}}


In [22]:
print(torch.exp(results['out_loss']))
print(results)

tensor(57231.2305, device='cuda:0')
{'out_loss': tensor(10.9549, device='cuda:0'), 'token_accuracy': {'accuracy': 0.7647058963775635, 'correct_tokens': 104, 'tot_tokens': 136}}


In [23]:
tokenizer.pad_token_type_id

0

In [24]:
exit()